# 第 7 週 Part A：補強版器材借用系統

本份筆記本是上週「極簡版系辦器材借用系統」的**補強版**。我們先**重現上週的錯誤**，再逐格看怎麼修。

| 上週的問題 | 本週修在哪裡 |
|---|---|
| 🔴 金鑰處理沒有防呆 | 第 1 格 |
| 🔴 AI 回覆多了符號就判斷錯（要重跑好幾次才成功） | 第 2 格重現、第 6 格修正 |
| 🔴 查不到資料時 `row[0]` 會當掉 | 第 3 格重現、第 8 格修正 |
| 🔴 提早 `return` 時資料庫連線沒關 | 第 8 格（`try / finally`） |
| 🔴 沒有借用人、沒有歸還功能 | 第 4 格（資料表）、第 8 格（歸還） |
| 🔴 呼叫 AI 失敗時整個程式當掉 | 第 6 格（`try / except`） |
| 🔴 只測一兩句就以為沒問題 | 第 7 格（測試清單） |

**開始前：** 左側 🔑 新增密鑰 `GROQ_API_KEY`，並打開「筆記本存取權」。

In [2]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


## 錯誤重現 1：AI 的回覆不一定乾乾淨淨

上週我們用 `strip()` 去掉前後空白，再檢查答案是否在清單裡。但 AI 有時會多回符號或文字，例如 `【單眼相機】`。
下面這格**不呼叫 AI**，直接模擬幾種可能的回覆，看看舊寫法和新寫法的差別。

In [6]:
# in
# str in str

print ( 'mation' in 'information')

# str in list

print ('mation' in ['information', 'management', 'mation'])

True
True


In [7]:
# ============================================================
# 第 2 格：🔴 錯誤重現 1 —— AI 回覆多了符號就判斷錯
# ============================================================
ITEMS = ["單眼相機", "腳架", "投影機"]
possible_ai_replies = ["單眼相機", "【單眼相機】", "單眼相機。", "答案：單眼相機", " 單眼相機\n"]

print("❌ 上週的寫法：strip() 之後，檢查「回覆」是否在清單內")
for reply in possible_ai_replies:
    ok = reply.strip() in ITEMS
    print(f"   AI 回 {reply!r} → {'認得' if ok else '誤判為「系統沒有這項器材」'}")

print()
print("✅ 補強的寫法：反過來檢查「清單裡哪一個名稱出現在回覆中」")
for reply in possible_ai_replies:
    found = "未知"
    for name in ITEMS:
        if name in reply:
            found = name
            break
    print(f"   AI 回 {reply!r} → {found}")

❌ 上週的寫法：strip() 之後，檢查「回覆」是否在清單內
   AI 回 '單眼相機' → 認得
   AI 回 '【單眼相機】' → 誤判為「系統沒有這項器材」
   AI 回 '單眼相機。' → 誤判為「系統沒有這項器材」
   AI 回 '答案：單眼相機' → 誤判為「系統沒有這項器材」
   AI 回 ' 單眼相機\n' → 認得

✅ 補強的寫法：反過來檢查「清單裡哪一個名稱出現在回覆中」
   AI 回 '單眼相機' → 單眼相機
   AI 回 '【單眼相機】' → 單眼相機
   AI 回 '單眼相機。' → 單眼相機
   AI 回 '答案：單眼相機' → 單眼相機
   AI 回 ' 單眼相機\n' → 單眼相機


## 錯誤重現 2：查不到資料時，`fetchone()` 會回傳 `None`

上週的 `current_status = row[0]` 假設一定查得到。萬一 AI 回了資料庫裡沒有的名稱，程式就會當掉。

In [8]:
# ============================================================
# 第 3 格：🔴 錯誤重現 2 —— row 是 None 時 row[0] 會當掉
# ============================================================
demo_conn = sqlite3.connect(":memory:")   # 暫時的記憶體資料庫，只做示範，不會產生檔案
demo_cur = demo_conn.cursor()
demo_cur.execute("CREATE TABLE items (name TEXT, status TEXT)")
demo_cur.execute("INSERT INTO items VALUES ('單眼相機', '可借用')")

demo_cur.execute("SELECT status FROM items WHERE name = ?", ("咖啡機",))
row = demo_cur.fetchone()
print("查詢「咖啡機」的結果：", row)

try:
    print(row[0])          # 上週的寫法：current_status = row[0]
except TypeError as e:
    print("❌ 程式當掉了：", e)

print("✅ 正確做法：先寫 if row is None: 處理查不到的情況，再取 row[0]")
demo_conn.close()

查詢「咖啡機」的結果： None
❌ 程式當掉了： 'NoneType' object is not subscriptable
✅ 正確做法：先寫 if row is None: 處理查不到的情況，再取 row[0]


## 建立資料庫（補強：借用人、歸還時間）

🔴 **為什麼每次都要 `DROP TABLE` 再重建？** 我批改時會在**全新的 Colab** 上從第一格跑到最後一格，
你的電腦裡的 `.db` 檔不會跟著交上去。所以資料庫必須由程式自己建立、自己放入初始資料。

In [11]:
# ============================================================
# 第 4 格：建立資料庫（每次執行都會重建，確保「從頭跑」結果一致）
# ============================================================
DB = "equipment.db"
ITEMS = ["單眼相機", "腳架", "投影機"]   # 系統認得的器材（AI 的答案只能是這些）
AVAILABLE = "可借用"                     # ⭐ 狀態文字集中定義，避免各處打錯字
BORROWED = "已借出"

conn = sqlite3.connect(DB)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS items")
cursor.execute("DROP TABLE IF EXISTS borrow_logs")

cursor.execute("""
    CREATE TABLE items (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        name   TEXT UNIQUE NOT NULL,      -- ⭐ UNIQUE：同名器材不能重複
        status TEXT NOT NULL
    )
""")

# ⭐ 補強：多了「借用人 borrower」與「歸還時間 return_time」
cursor.execute("""
    CREATE TABLE borrow_logs (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        item_name   TEXT NOT NULL,
        borrower    TEXT NOT NULL,
        borrow_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        return_time TIMESTAMP             -- 還沒歸還時是空的（NULL）
    )
""")

cursor.executemany(
    "INSERT INTO items (name, status) VALUES (?, ?)",
    [("單眼相機", AVAILABLE), ("腳架", BORROWED), ("投影機", AVAILABLE)]
)

conn.commit()
conn.close()
print("✅ 資料庫建立完成")

✅ 資料庫建立完成


In [13]:
# ============================================================
# 第 5 格：顯示資料表（之後每次操作完都可以呼叫，確認資料庫真的有變）
# ============================================================
def show_tables():
    conn = sqlite3.connect(DB)
    print("📦 items（器材）")
    display(pd.read_sql_query("SELECT * FROM items", conn))
    print("📝 borrow_logs（借用紀錄）")
    display(pd.read_sql_query("SELECT * FROM borrow_logs", conn))
    conn.close()

show_tables()

📦 items（器材）


,id,name,status
0,1,單眼相機,可借用
1,2,腳架,已借出
2,3,投影機,可借用


📝 borrow_logs（借用紀錄）


,id,item_name,borrower,borrow_time,return_time


## AI 解讀：一次判斷「動作」和「器材」

補強重點有三個：prompt 裡給**格式與範例**、呼叫失敗時**不當掉**、**不直接相信** AI 的回覆。
函式會回傳三個值：動作、器材，以及 AI 的原始回覆（方便除錯）。

In [14]:
# ============================================================
# 第 6 格：AI 解讀使用者的話 → (動作, 器材, 原始回覆)
# ============================================================
ACTIONS = ["借用", "歸還"]
item_list = "、".join(ITEMS)

def ai_parse(user_say):
    prompt = f"""你是學校系辦的器材借用助理。
系統只有這些器材：{item_list}。
請判斷使用者想「借用」還是「歸還」，以及是哪一項器材。

只能回覆一行，格式為：動作,器材
- 動作只能是：借用、歸還、未知
- 器材只能是：{item_list}、未知
- 不要回覆任何其他文字

範例：
使用者：我想借投影機 → 借用,投影機
使用者：腳架用完了，拿來還 → 歸還,腳架
使用者：今天午餐吃什麼 → 未知,未知

使用者：{user_say}"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        raw = response.choices[0].message.content or ""   # ⭐ content 偶爾會是 None
    except Exception as e:
        # ⭐ 補強：網路不穩、金鑰錯誤、超過使用量時，不要讓整個程式當掉
        print("⚠️ 呼叫 AI 失敗：", e)
        return "未知", "未知", ""

    # ⭐ 補強：不直接相信 AI 的格式，自己從回覆中「找」出允許的答案
    action = "未知"
    for a in ACTIONS:
        if a in raw:
            action = a
            break

    item = "未知"
    for name in ITEMS:
        if name in raw:
            item = name
            break

    return action, item, raw

## 測試清單：先確認 AI 解讀得對，再接資料庫

每一列是 `(使用者說的話, 預期動作, 預期器材)`。`"*"` 表示「這一項不在意」。
好的測試清單要涵蓋三種情況：**正常**（該成功的）、**錯誤**（該被拒絕的）、**邊界**（容易出錯的奇怪說法）。

In [20]:
# ============================================================
# 第 7 格：測試清單
# ============================================================
test_list = [
    ("助教好，我下午要拍影片，想借一台相機", "借用", "單眼相機"),   # 正常：說法不完全一樣
    ("投影機用完了，我拿來還",               "歸還", "投影機"),     # 正常：歸還
    ("我要借腳架",                           "借用", "腳架"),       # 正常：最簡單的說法
    ("請問可以借咖啡機嗎",                   "*",    "未知"),       # 錯誤：沒有這項器材
    ("今天午餐吃什麼",                       "未知", "未知"),       # 錯誤：無關的話
    ("我想借腳架，順便還投影機",             "*",    "*"),          # 邊界：一句話兩件事（觀察用）
]

passed = 0
for say, exp_action, exp_item in test_list:
    action, item, raw = ai_parse(say)
    ok = (exp_action in ("*", action)) and (exp_item in ("*", item))
    passed += ok
    print(f"{'✅' if ok else '❌'} {say}")
    print(f"     AI 原始回覆：{raw!r}")
    print(f"     系統解讀：{action} / {item}（預期：{exp_action} / {exp_item}）\n")

print(f"通過 {passed} / {len(test_list)} 句")

✅ 助教好，我下午要拍影片，想借一台相機
     AI 原始回覆：'借用,單眼相機'
     系統解讀：借用 / 單眼相機（預期：借用 / 單眼相機）

✅ 投影機用完了，我拿來還
     AI 原始回覆：'歸還,投影機'
     系統解讀：歸還 / 投影機（預期：歸還 / 投影機）

✅ 我要借腳架
     AI 原始回覆：'借用,腳架'
     系統解讀：借用 / 腳架（預期：借用 / 腳架）

✅ 請問可以借咖啡機嗎
     AI 原始回覆：'借用,未知'
     系統解讀：借用 / 未知（預期：* / 未知）

✅ 今天午餐吃什麼
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

✅ 我想借腳架，順便還投影機
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：* / *）

通過 6 / 6 句


## 借用與歸還（補強：檢查 None、一定關閉連線、新增歸還）

`try / finally` 的意思是：**不管從哪一行 `return` 離開，`finally` 裡的程式都一定會執行**。
所以把 `conn.close()` 放在 `finally`，就不會再忘記關連線。

In [21]:
# ============================================================
# 第 8 格：借用與歸還（直接操作資料庫，不呼叫 AI）
# ============================================================
def borrow_item(item, borrower):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT status FROM items WHERE name = ?", (item,))
        row = cursor.fetchone()

        if row is None:                                   # ⭐ 補強：查不到資料
            return f"系統沒有「{item}」這項器材。"
        if row[0] == BORROWED:
            return f"不好意思，{item} 已經被借走了，請改借其他器材。"

        cursor.execute("UPDATE items SET status = ? WHERE name = ?", (BORROWED, item))
        cursor.execute("INSERT INTO borrow_logs (item_name, borrower) VALUES (?, ?)", (item, borrower))
        conn.commit()
        return f"🎉 {borrower} 借用「{item}」成功！"
    finally:
         conn.close()                                     # ⭐ 補強：一定會關閉連線


def return_item(item, borrower):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT status FROM items WHERE name = ?", (item,))
        row = cursor.fetchone()

        if row is None:
            return f"系統沒有「{item}」這項器材。"
        if row[0] == AVAILABLE:
            return f"「{item}」目前沒有被借出，不需要歸還。"

        cursor.execute("UPDATE items SET status = ? WHERE name = ?", (AVAILABLE, item))
        # 把這項器材「最新一筆、還沒歸還」的紀錄填上歸還時間
        cursor.execute("""
            UPDATE borrow_logs SET return_time = CURRENT_TIMESTAMP
            WHERE id = (SELECT MAX(id) FROM borrow_logs
                        WHERE item_name = ? AND return_time IS NULL)
        """, (item,))
        conn.commit()
        return f"✅「{item}」已歸還，謝謝 {borrower}！"
    finally:
        conn.close()

In [22]:
# ============================================================
# 第 9 格：主流程 —— 檢查輸入 → AI 解讀 → 借用或歸還
# ============================================================
def borrow_system(user_say, borrower):
    # ⭐ 補強：先檢查輸入，空白的就不要浪費一次 AI 呼叫
    if not borrower or not borrower.strip():
        return "請先輸入你的姓名。"
    if not user_say or not user_say.strip():
        return "請輸入你想借用或歸還的器材。"

    action, item, raw = ai_parse(user_say)
    print(f"📌 [AI 解讀] 動作={action}，器材={item}（原始回覆：{raw!r}）")

    if item == "未知":
        return f"抱歉，系統目前只有：{item_list}。"
    if action == "借用":
        return borrow_item(item, borrower.strip())
    if action == "歸還":
        return return_item(item, borrower.strip())
    return "抱歉，我不確定你要借用還是歸還，請再說清楚一點。"

In [25]:
# ============================================================
# 第 10 格：完整流程測試（請對照每一行後面的「預期」）
# ============================================================
cases = [
    ("我想借單眼相機拍作業",     "王小明"),   # 預期：借用成功
    ("我也想借單眼相機",         "陳小華"),   # 預期：已經被借走
    ("單眼相機用完了，拿來還",   "王小明"),   # 預期：歸還成功
    ("我要還投影機",             "陳小華"),   # 預期：沒有被借出，不需要歸還
    ("我要借投影機",             ""),         # 預期：請先輸入姓名
]
for say, name in cases:
    print("-" * 50)
    print(f"🙋 {name or '（未填姓名）'}：{say}")
    print(borrow_system(say, name))

print("-" * 50)
show_tables()

--------------------------------------------------
🙋 王小明：我想借單眼相機拍作業
📌 [AI 解讀] 動作=借用，器材=單眼相機（原始回覆：'借用,單眼相機'）
🎉 王小明 借用「單眼相機」成功！
--------------------------------------------------
🙋 陳小華：我也想借單眼相機
📌 [AI 解讀] 動作=借用，器材=單眼相機（原始回覆：'借用,單眼相機'）
不好意思，單眼相機 已經被借走了，請改借其他器材。
--------------------------------------------------
🙋 王小明：單眼相機用完了，拿來還
📌 [AI 解讀] 動作=歸還，器材=單眼相機（原始回覆：'歸還,單眼相機'）
✅「單眼相機」已歸還，謝謝 王小明！
--------------------------------------------------
🙋 陳小華：我要還投影機
📌 [AI 解讀] 動作=歸還，器材=投影機（原始回覆：'歸還,投影機'）
「投影機」目前沒有被借出，不需要歸還。
--------------------------------------------------
🙋 （未填姓名）：我要借投影機
請先輸入你的姓名。
--------------------------------------------------
📦 items（器材）


,id,name,status
0,1,單眼相機,可借用
1,2,腳架,已借出
2,3,投影機,可借用


📝 borrow_logs（借用紀錄）


,id,item_name,borrower,borrow_time,return_time
0,1,單眼相機,王小明,2026-10-29 02:22:32,2026-10-29 02:22:33
1,2,單眼相機,王小明,2026-10-29 02:23:07,2026-10-29 02:23:08
2,3,單眼相機,王小明,2026-10-29 02:23:12,2026-10-29 02:23:13


## 網頁介面

🔴 **提醒：** Gradio 的 share 連結**一段時間後就會失效**，存到 GitHub 後也**看不到這個網頁**。
所以老師批改看的是**上面第 7、10 格的執行結果**，網頁只是讓你自己操作用的。

In [26]:
# ============================================================
# 第 11 格：Gradio 網頁（放在最後一格）
# ============================================================
demo = gr.Interface(
    fn=borrow_system,                        # 輸入框的順序 = 函式參數的順序
    inputs=[
        gr.Textbox(label="💬 借用或歸還需求（例如：我想借投影機）"),
        gr.Textbox(label="🙋 你的姓名"),
    ],
    outputs=gr.Textbox(label="🤖 系統回應"),
    title="🏫 系辦器材借用系統（補強版）",
    description="結合 Groq AI 語意辨識與 SQLite 資料庫（可測試：單眼相機、腳架、投影機；可借用也可歸還）",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://650c78b2a376ef4cd9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
show_tables()